# 02 EDA Base Tickets

Este notebook toma el `Parquet` de la etapa `01`, prepara la base para analisis exploratorio y genera el `Parquet` oficial de la etapa `02`.


## 1. Librerias


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Definir rutas


In [ ]:
from pathlib import Path

def resolve_project_root() -> Path:
    # Buscar la raiz del proyecto a partir del directorio actual.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "parquets").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto")

PROJECT_ROOT = resolve_project_root()
PROJECT_ROOT

# Definir el parquet de entrada y la salida de la etapa 02.
INPUT_PATH = PROJECT_ROOT / "parquets" / "01_Carga_y_Validacion_Parquet" / "01_base_tickets_modelado.parquet"
OUTPUT_DIR = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets"
OUTPUT_PATH = OUTPUT_DIR / "02_base_eda_tickets.parquet"

INPUT_PATH, OUTPUT_PATH


## 3. Cargar la base de entrada


In [ ]:
# Leer la base principal generada en la etapa 01.
df = pd.read_parquet(INPUT_PATH)

# Mostrar las primeras filas del dataset base.
df.head()


### Resultado breve

La base de entrada ya representa tickets consolidados y sirve como punto de partida para el analisis exploratorio.


## 4. Crear variables derivadas para EDA


In [ ]:
# Generar columnas auxiliares para revisar calendario y consistencia.
df["anio_mes"] = pd.to_datetime(df["fecha"]).dt.strftime("%Y-%m")
df["dia_tipo"] = df["fin_semana"].map({True: "Fin de semana", False: "Entre semana"})
df["residuo_subtotal_total"] = df["subtotal_ticket"] - df["total_pedido"]
df["residuo_abs_subtotal_total"] = df["residuo_subtotal_total"].abs()
df["residuo_pago_total"] = df["monto_pago"] - df["total_pedido"]
df["ticket_consistente_subtotal"] = (df["residuo_abs_subtotal_total"] <= 0.01).astype(int)
df["ticket_consistente_pago"] = (df["residuo_pago_total"].abs() <= 0.01).astype(int)
df["rango_total_pedido"] = pd.cut(
    df["total_pedido"],
    bins=[0, 300, 520, float("inf")],
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True,
)

df.head()


### Resultado breve

En esta etapa se agregan variables de apoyo para entender mejor la calidad del dato, la consistencia de montos y la distribucion general de los tickets.


## 5. Revisar consistencia y distribuciones


In [ ]:
# Resumir la consistencia observada en subtotales y pagos.
df[["ticket_consistente_subtotal", "ticket_consistente_pago"]].sum()


In [ ]:
# Graficar la distribucion del total del pedido y una comparacion por tipo de dia.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["total_pedido"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribucion de total_pedido")
axes[0].set_xlabel("total_pedido")

sns.boxplot(data=df, x="dia_tipo", y="total_pedido", ax=axes[1])
axes[1].set_title("Total del pedido por tipo de dia")
axes[1].set_xlabel("dia_tipo")
axes[1].set_ylabel("total_pedido")
plt.tight_layout()


### Resultado breve

Las visualizaciones permiten revisar si el monto del ticket cambia entre dias y si existen valores extremos que despues afecten a los modelos.


## 6. Exportar el parquet de la etapa 02


In [ ]:
# Crear la carpeta de salida si todavia no existe.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Definir el orden final de columnas de la base EDA.
columnas_ordenadas = [
    "id_ticket_modelado", "fecha", "anio_mes", "dia", "mes", "nombre_mes", "trimestre", "anio",
    "dia_semana", "dia_tipo", "fin_semana", "id_cliente", "id_sucursal", "ciudad", "capacidad_sucursal",
    "id_empleado", "tipo_empleado", "salario", "turno", "id_mesa", "numero_mesa", "capacidad_mesa",
    "id_metodo_pago", "metodo_pago", "lineas_ticket", "cantidad_total", "platillos_distintos",
    "categorias_distintas", "subtotal_ticket", "total_pedido", "monto_pago", "diferencia_pago",
    "residuo_subtotal_total", "residuo_abs_subtotal_total", "residuo_pago_total", "ticket_consistente_subtotal",
    "ticket_consistente_pago", "incluye_bebida", "incluye_postre", "incluye_entrada", "incluye_platillo_fuerte",
    "ticket_alto", "rango_total_pedido"
]

df_eda = df[columnas_ordenadas].copy()
df_eda.to_parquet(OUTPUT_PATH, index=False)
print(f"Parquet generado en: {OUTPUT_PATH}")


In [ ]:
# Verificar que el parquet exportado se pueda leer correctamente.
df_eda_exportado = pd.read_parquet(OUTPUT_PATH)
df_eda_exportado.shape


### Resultado breve

La etapa `02` queda lista cuando el parquet `02_base_eda_tickets.parquet` ya contiene la base enriquecida y se puede usar como entrada directa para los modelos.
